# نماذج التعلم الأساسية شبه الخاضعة للإشراف



في بعض الأحيان لا يكون الضغط على Enter في TensorFlow كافيًا للحصول على نتائج جيدة. في بعض الأحيان يكون لديك القليل جدًا من البيانات للعمل بها، ويمنحك تقدير الاحتمالية القصوى بعض الإجابات الغريبة. أو حتى أكثر شيوعًا - البيانات والتصنيفات *باهظة الثمن* والعلاقة بين عدد الحالات في مجموعة البيانات الخاصة بك وجودة النموذج النهائي ليست خطية على الإطلاق.
وذلك عندما يأتي دور التعلم شبه الخاضع للإشراف (SSL).
 
 > التعلم شبه الخاضع للإشراف هو فئة من مهام وتقنيات التعلم الآلي التي تستخدم أيضًا البيانات غير المسماة للتدريب - عادةً ما تكون كمية صغيرة من البيانات المصنفة مع كمية كبيرة من البيانات غير المسماة [[المصدر]](https://en.wikipedia.org/wiki/Semi-supervised_learning).
 
تستخدم هذه الأنواع من خوارزميات ML كلاً من البيانات المصنفة وغير المسماة للحصول على تنبؤات أكثر دقة (أو في حالات نادرة للحصول عليها على الإطلاق).  



<center><p><img src="@@KEEP_00027@@ title="Types of learning"/></p></center>  
Img 1 - أنواع التعلم [[المصدر]](https://www.slideshare.net/Dataiku/dataiku-hadoop-summit-semisupervised-learning-with-hadoop-for-understanding-user-web-behaviours)



وبما أننا نتعامل مع بيانات غير مصنفة (من الواضح أنها حالات غير معروفة)، فإننا نستخلص بعض الافتراضات من التعلم غير الخاضع للإشراف:
 
 - افتراض الاستمرارية (*النقاط القريبة من بعضها البعض أكثر عرضة لمشاركة التسمية*)
 - افتراض المجموعة (*تميل البيانات إلى تشكيل مجموعات منفصلة، ومن المرجح أن تتشارك النقاط الموجودة في نفس المجموعة في التسمية*)
 - افتراض المتشعب (* تقع البيانات تقريبًا على مشعب ذي بُعد أقل بكثير من مساحة الإدخال*)



تبدو معقولة بدرجة كافية على الرغم من أنه لا يمثل مشكلة في العثور على حجج مضادة.
لماذا نحتاج إلى كل هذه التعقيدات؟ حسنا، لأنه يعمل (من الناحية النظرية).


<center><p><img src="@@KEEP_00029@@ title="SSL explanation"/></p></center>
    
Img 2 - شرح شبه خاضع للإشراف [[المصدر]](http://people.cs.uchicago.edu/~niyogi/papersps/BNSJMLR.pdf)



إن فكرة SSL بأكملها هي فكرة بايزي بطبيعتها، لكننا لن نتعمق فيها.
على سبيل القياس لـ SSL، توفر بعض الموارد (https://deepai.org/machine-learning-glossary-and-terms/semi-supervised-learning) منطقًا استقرائيًا بشريًا: نحن نعرف فقط بعض أمثلة هذه الظاهرة، لكننا نحاول استنتاج المبادئ العامة ونفس الشيء هنا: تستخدم الخوارزميات عددًا قليلاً من النقاط المرجعية (بياناتنا المُصنفة) للعثور على النمط العام الذي يناسب بياناتنا غير المُصنفة بشكل أفضل. ولكن هذا ليس هو الحال دائما. ستحاول بعض الخوارزميات العثور على تعميم للبيانات المقدمة (استنتج دالة تصف بياناتنا بشكل أفضل)، لكن بعض الخوارزميات ستستخدم شيئًا يسمى [التعلم النقلي](https://en.wikipedia.org/wiki/Transduction_(machine_learning)) ([المزيد هنا](http://www.cs.cornell.edu/courses/cs4780/2009fa/lecture/13-transduction.pdf)). سوف نستخدم النماذج التي تتبع هذا النهج لحل مهمة التصنيف. 
بشكل عام، يحاول التعلم الاستقرائي الحصول على تعميم من البيانات المقدمة وعندها فقط التنبؤ بالبيانات الجديدة. سيحاول التعلم الانتقالي فقط التنبؤ بالبيانات الجديدة في ضوء بيانات التدريب، وتخطي جزء التعميم.
دعونا رمز قليلا. سأضيف تعليقات وأشرح ما أفعله. أعتقد أنه لن يكون ضروريًا في معظم الأوقات، ولكن لا يزال.


In [ ]:
# Basic imports
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.utils import shuffle


يحتوي `sklearn` على خوارزميتين SSL: نشر الملصقات ونشر الملصقات. دعونا استيرادها.
سنستخدم أيضًا [`pomegranate`](https://pomegranate.readthedocs.io/en/latest/index.html). إنها ليب بايزي بها الكثير من الميزات ولكننا سنأخذ منها نموذجًا أو نموذجين فقط. إنه مكتوب بما يسمى تدوين skl - يستخدم نفس بناء الجملة وأسماء الأساليب مثل `sklearn`، لذلك ستفهمه بسرعة كبيرة. لتثبيته تشغيل:
من خلال النقطة
> نقطة تثبيت الرمان
من خلال اناكوندا
> كوندا تثبيت الرمان


In [ ]:
import pomegranate as pg
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.semi_supervised import LabelPropagation, LabelSpreading

In [ ]:
import warnings

warnings.simplefilter("ignore")  # we don't wanna see that
np.random.seed(
    1
)  # i'm locking seed at the begining since we will use some heavy RNG stuff, be aware

سوف نستخدم [مجموعة البيانات](https://scikit-learn.org/stable/datasets/index.html#breast-cancer-dataset). هنا `sklearn` [المحمل](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)
الصفات:
- رقم الهوية 
- التشخيص (M = خبيث، B = حميد) 
يتم حساب عشر ميزات ذات قيمة حقيقية لكل نواة خلية: 
- نصف القطر (متوسط المسافات من المركز إلى النقاط الموجودة على المحيط) 
- الملمس (الانحراف المعياري لقيم التدرج الرمادي) 
- محيط 
- المنطقة 
- النعومة (الاختلاف المحلي في أطوال نصف القطر) 
- الاكتناز (المحيط ^ 2 / المساحة - 1.0) 
- تقعر (شدة الأجزاء المقعرة من الكفاف) 
- النقاط المقعرة (عدد الأجزاء المقعرة من الكفاف) 
- التماثل 
- البعد الكسري ("تقريب الساحل" - 1)
تم حساب المتوسط والخطأ المعياري و"الأسوأ" أو الأكبر (متوسط القيم الثلاث الكبرى) لهذه الميزات لكل صورة، مما أدى إلى 30 ميزة. على سبيل المثال، الحقل 3 هو نصف القطر المتوسط، والحقل 13 هو نصف القطر SE، والحقل 23 هو نصف القطر الأسوأ.
يتم إعادة ترميز جميع قيم الميزات بأربعة أرقام مهمة.
قيم السمات المفقودة: لا شيء
التوزيع الطبقي: 357 حميد، 212 خبيث


In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data["target"]

In [ ]:
df.head()


الآن سوف ننظر بإيجاز في مجموعة البيانات الخاصة بنا


In [ ]:
df.info()

In [ ]:
df.describe()


سنقوم بخلط مجموعة البيانات لدينا لأنها تحتوي على نظام ونحن لا نريد ذلك. 
أيضًا، سأقوم بتقليل الأبعاد عن طريق إسقاط الميزات لتبسيط كل شيء.
بعد ذلك، سنقوم بإنشاء `X` و`Y` وتقسيمهما إلى ثلاثة أجزاء: بيانات القطار المُسمى (1)، وبيانات القطار غير المُسمى (2)، وبيانات الاختبار (3). سوف نقوم بإسقاط الكثير من الميزات (se وأسوأ الأعمدة؛ المنطقة/المحيط زائدة عن الحاجة لأنها مرتبطة بشكل كبير بنصف القطر؛ ميزة النقاط المقعرة زائدة عن الحاجة أيضًا وترتبط بالتقعر والاكتناز).تحذير بسيط: لا يمكن تحقيق إعادة إنتاج النتائج إلا إذا قمت بخلط البيانات من ترتيبها الأصلي (الترتيب الموجود في ملف CSV). إذا حاولت التبديل العشوائي من حالة أخرى (التبديل العشوائي بعد أن قمت بالفعل بخلطه، وما إلى ذلك) فسوف يعطيك نتائج مختلفة.


In [ ]:
df = shuffle(df, random_state=1)
X = df.drop(
    [
        "target",
        "radius error",
        "texture error",
        "perimeter error",
        "area error",
        "smoothness error",
        "compactness error",
        "concavity error",
        "concave points error",
        "symmetry error",
        "fractal dimension error",
        "worst radius",
        "worst texture",
        "worst perimeter",
        "worst area",
        "worst smoothness",
        "worst compactness",
        "worst concavity",
        "worst concave points",
        "worst symmetry",
        "worst fractal dimension",
        "mean area",
        "mean perimeter",
        "mean concave points",
    ],
    axis=1,
)
y = df["target"]


نظرًا لأن لدينا سبع ميزات فقط الآن، يمكننا القيام بشيء يسمى [pairplot](https://seaborn.pydata.org/generated/seaborn.pairplot.html): حيث سيتم رسم مخططات نقطية بين كل ميزة وتوزيعها بشكل قطري. سيساعدنا هذا في العثور على الارتباطات ومعرفة كيف يمكننا تطبيع ميزاتنا.
وهذا سوف يستغرق وقتا. المزيد من الميزات والكائنات = المزيد من الوقت، لذا لا تجرب ذلك على مجموعات بيانات كبيرة. إذا كنت تريد، يمكنك تجربتها على مجموعة البيانات الكاملة `df` (سيستغرق الأمر وقتًا أطول وسيكون كل رسم بياني أصغر)، وسترى ميزات مترابطة للغاية أسقطتها أعلاه.


In [ ]:
sns.pairplot(X)
# sns.pairplot(df)


الآن سنقوم بدمج البيانات المصنفة وغير المسماة.
الممارسة الشائعة (معظم المكتبات ستحتاج إليها بهذه الطريقة) هي تسمية البيانات غير المسماة `-1`. لن تقبل بعض libs سوى `NaN` كتسمية للبيانات غير المسماة. تحقق دائمًا من الوثائق وأدلة المستخدم.


In [ ]:
X_1, X_2, X_3 = np.split(X, [int(0.1 * len(X)), int(0.5 * len(X))])
y_1, y_2, y_3 = np.split(y, [int(0.1 * len(y)), int(0.5 * len(y))])
y_1_2 = np.concatenate((y_1, y_2.apply(lambda x: -1)))
X_1_2 = np.concatenate((X_1, X_2))

In [ ]:
index = ["Algorithm", "ROC AUC"]
results = pd.DataFrame(columns=index)

In [ ]:
logreg = LogisticRegression(random_state=1, class_weight="balanced")
logreg.fit(X_1, y_1)
results = results.append(
    pd.Series(
        ["Logistic Regression", roc_auc_score(y_3, logreg.predict_proba(X_3)[:, 1])],
        index=index,
    ),
    ignore_index=True,
)
results


فقط للإشارة إلى توقيت LR.
*تذكير: df عبارة عن مصفوفة كثيفة مقاس 569 × 31*


In [ ]:
%%timeit
logreg_test = LogisticRegression(random_state=1, class_weight="balanced")
logreg_test.fit(df, y)
logreg_test.predict_proba(df);


## نشر التسمية



حان الوقت لاستخدام نموذج SSL الأول لدينا - `LabelPropagation` (LP). انها بسيطة جدا. بشكل بديهي، تبدو وكأنها خوارزمية تجميعية تعمل فقط على "نشر" التصنيفات إلى أقرب نقاط البيانات. ولكن علينا أن نذهب أعمق قليلا.
Y هي مصفوفة التسمية الخاصة بنا، ويبلغ حجمها $(l+u)\times C$، حيث l - مقدار نقاط البيانات المسماة، u - مقدار نقاط البيانات غير المسماة، C - عدد الفئات. لذلك في مهمة التصنيف الثنائي سنحصل على عمودين. إنها نوع من القائمة التي سنحصل عليها من طريقة `predict_proba`.ستقوم الخوارزمية بإنشاء رسم بياني متصل بالكامل حيث تكون العقد **جميع** نقاط البيانات. ستكون الحواف بين العقد أوزانًا $w_{i,j}$: 
$$w_{i,j} = exp(\frac{d_{i,j}^2}{\sigma^2})$$
حيث *d* - دالة المسافة (الإقليدية في هذه الحالة، ولكن بشكل عام يمكن أن تكون أي دالة مسافة تريدها)، $\sigma$ - معلمة مفرطة، تتحكم في (تقليص) الأوزان.
ثم نقوم ببناء مصفوفة الانتقال الاحتمالية T: 
$$T_{i,j} = \frac{w_{i,j}}{\sum_{k=1}^{l+u}w_{i,j}}$$
T هي مجرد مصفوفة مع احتمال أن تكون كل نقطة بيانات في الفئة C. نقاط البيانات الموسومة لدينا لديها احتمال 1.0 لتكون في الفئة C (نظرًا لأننا نعرف فئاتها) وستحصل البيانات غير المسماة على فئاتها من الجيران (قمنا بحساب الأوزان سابقًا، فهي تعتمد على المسافة بين نقطتين).
تبدو الخوارزمية بأكملها كما يلي:
1. نشر Y <- TY (نقوم "بنشر" التسميات من البيانات المصنفة إلى البيانات غير المسماة)
2. تطبيع الصف Y (قيمة العنصر في الصف / مجموع كل قيم العناصر في الصف)
3. قم بتثبيت البيانات المُصنفة (نقوم بإصلاح بياناتنا المُصنفة، بحيث لا تغير الخوارزمية احتمالية الفئة، أو بمعنى آخر تغير التسمية)
4. كرر من الخطوة 1 حتى تتقارب Y (نعيد حساب المسافات والأوزان، مما سيعطينا مصفوفة انتقالية مختلفة ستغير اعتقادنا في التسميات المخصصة، كرر حتى تتقارب العملية).
في حالة تنفيذ `sklearn` لدينا خيار وظيفة الترجيح: RBF (انظر صيغة $w_{i,j}$) أو KNN ($1(x' \in kNN(x))$). KNN أسرع ويعطي تمثيلًا متناثرًا. من الصعب حساب وتخزين مصفوفة الانتقال الكثيفة في الذاكرة RBF، ولكن لديها المزيد من الخيارات للضبط. تعلم أيضًا تنفيذ RBF بدلاً من القسمة على $\sigma^2$ وضربه بـ `gamma`، لذلك يجب أن يكون عائمًا وليس عددًا صحيحًا.لمعرفة المزيد، يمكنك قراءة [هذا](http://pages.cs.wisc.edu/~jerryzhu/pub/CMU-CALD-02-107.pdf) و[هذا](https://scikit-learn.org/stable/modules/label_propagation.html#label-propagation). يحتوي `sklearn` أيضًا على بعض الأمثلة الرائعة لـ `LabelPropagation`: [SVM vs LP](https://scikit-learn.org/stable/auto_examples/semi_supervised/plot_label_propagation_versus_svm_iris.html#sphx-glr-auto-examples-semi-supervised-plot-label-propagation-versus-svm-iris-py)، [LP التجريبي](https://scikit-learn.org/stable/auto_examples/semi_supervised/plot_label_propagation_structure.html#sphx-glr-auto-examples-semi-supervised-plot-label-propagation-structure-py)، [التعرف على أرقام LP](https://scikit-learn.org/stable/auto_examples/semi_supervised/plot_label_propagation_digits.html#sphx-glr-auto-examples-semi-supervised-plot-label-propagation-digits-py) و[LP أرقام نشطة التعلم](https://scikit-learn.org/stable/auto_examples/semi_supervised/plot_label_propagation_digits_active_learning.html#sphx-glr-auto-examples-semi-supervised-plot-label-propagation-digits-active-learning-py).



الآن سوف أقوم بتعريف وظيفة صغيرة من شأنها أن تعطينا مخططًا لـ ROC AUC لخوارزميتنا اعتمادًا على قائمة النواة والمعلمات الخاصة بنا.


In [ ]:
def label_prop_test(kernel, params_list, X_train, X_test, y_train, y_test):
    plt.figure(figsize=(20, 10))
    n, g = 0, 0
    roc_scores = []
    if kernel == "rbf":
        for g in params_list:
            lp = LabelPropagation(
                kernel=kernel, n_neighbors=n, gamma=g, max_iter=100000, tol=0.0001
            )
            lp.fit(X_train, y_train)
            roc_scores.append(roc_auc_score(y_test, lp.predict_proba(X_test)[:, 1]))
    if kernel == "knn":
        for n in params_list:
            lp = LabelPropagation(
                kernel=kernel, n_neighbors=n, gamma=g, max_iter=100000, tol=0.0001
            )
            lp.fit(X_train, y_train)
            roc_scores.append(roc_auc_score(y_test, lp.predict_proba(X_test)[:, 1]))
    plt.figure(figsize=(16, 8))
    plt.plot(params_list, roc_scores)
    plt.title("Label Propagation ROC AUC with " + kernel + " kernel")
    plt.show()
    print("Best metrics value is at {}".format(params_list[np.argmax(roc_scores)]))

In [ ]:
gammas = [9e-6, 1e-5, 2e-5, 3e-5, 4e-5, 5e-5, 6e-5, 7e-5, 8e-5, 9e-5]
label_prop_test("rbf", gammas, X_1_2, X_3, y_1_2, y_3)

In [ ]:
ns = np.arange(50, 60)
label_prop_test("knn", ns, X_1_2, X_3, y_1_2, y_3)


يمكنك الآن تحديد النموذج الخاص بك بشكل منفصل باستخدام أفضل قيمة للمعلمة الفائقة (أو أيًا كانت) والتحقق من درجاتها


In [ ]:
lp_rbf = LabelPropagation(kernel="rbf", gamma=9e-6, max_iter=100000, tol=0.0001)
lp_rbf.fit(X_1_2, y_1_2)
results = results.append(
    pd.Series(
        ["Label Propagation RBF", roc_auc_score(y_3, lp_rbf.predict_proba(X_3)[:, 1])],
        index=index,
    ),
    ignore_index=True,
)

lp_knn = LabelPropagation(kernel="knn", n_neighbors=53, max_iter=100000, tol=0.0001)
lp_knn.fit(X_1_2, y_1_2)
results = results.append(
    pd.Series(
        ["Label Propagation KNN", roc_auc_score(y_3, lp_knn.predict_proba(X_3)[:, 1])],
        index=index,
    ),
    ignore_index=True,
)

In [ ]:
results


الآن دعونا نتحدث عن الوقت. يمكنك فقط إلقاء نظرة على النتائج أو تشغيل الأمر على جهازك. في المصفوفات الكثيفة يكون الأمر جيدًا، ولكن في التمثيل المتناثر يكون الأمر كارثيًا (ربما يكون knn kernel على ما يرام). ولكن لهذا لدينا [نشر التسمية المتفرقة](https://arxiv.org/abs/1612.01414)


In [ ]:
%%timeit
rbf_lp_test = LabelPropagation(kernel="rbf")
rbf_lp_test.fit(df, y)
rbf_lp_test.predict_proba(df);

In [ ]:
%%timeit
knn_lp_test = LabelPropagation(kernel="knn")
knn_lp_test.fit(df, y)
knn_lp_test.predict_proba(df);


## انتشار التسمية



التالي هو `LabelSpreading` (LS). الخوارزمية تشبه إلى حد كبير خوارزمية التجميع الطيفية *خوارزمية القطع المقيسة* ([انظر هنا](https://en.wikipedia.org/wiki/Spectral_clustering)، [هنا](https://towardsdatascience.com/spectral-clustering-for-beginners-d08b7d25b4d8) و[هنا](https://papers.nips.cc/paper/2092-on-spectral-clustering-analysis-and-an-algorithm.pdf)).
سيقوم LS بإنشاء مصفوفة تقارب (مثل خطوة حساب أوزان LP): 
$$W_{i,j} = exp(\frac{-||x_i - x_j||^2}{\sigma^2})$$
لكل $i\neq j$ و$W_{i,j} = 0$ لكل $i = j$
ثم سنقوم ببناء المصفوفة (لابلاسيان):
$$S = D^{-1/2}WD^{−1/2}$$
حيث D - مصفوفة قطرية بعنصر *(i,i)* يساوي مجموع الصف i من W.
هاتان الخطوتان هما مجرد تجميع طيفي لجميع بياناتنا. التاليان أكثر إثارة للاهتمام:
 
كرر $F(t+1) = αSF(t)+(1−α)Y$ حتى التقارب، حيث α هي معلمة في (0، 1)، F(t) - وظيفة التصنيف
دع $F^*$ يشير إلى حد التسلسل {F(t)}. قم بتسمية كل نقطة $x_i$ كتسمية $y_i = argmax≤F^*_{i,j}$خلال كل تكرار للخطوة الثالثة، تتلقى كل نقطة المعلومات من جيرانها (الحد الأول)، وتحتفظ أيضًا بمعلوماتها الأولية (الحد الثاني). تحدد المعلمة α المقدار النسبي للمعلومات الواردة من جيرانها ومعلومات التسمية الأولية الخاصة بها. أخيرًا، يتم تعيين تسمية كل نقطة غير مسماة لتكون الفئة التي تلقت معظم المعلومات عنها أثناء عملية التكرار.
يمكنك قراءة مقالة حول نشر الملصقات [هنا](http://citeseer.ist.psu.edu/viewdoc/download;jsessionid=19D11D059FCEDAFA443FB135B4065A6A?doi=10.1.1.115.3219&rep=rep1&type=pdf)، `sklearn`، مستندات المستخدم موجودة [هنا](https://scikit-learn.org/stable/modules/generated/sklearn.semi_supervised.LabelSpreading.html).


In [ ]:
def labels_spread_test(kernel, hyperparam, alphas, X_train, X_test, y_train, y_test):
    plt.figure(figsize=(20, 10))
    n, g = 0, 0
    roc_scores = []
    if kernel == "rbf":
        g = hyperparam
    if kernel == "knn":
        n = hyperparam
    for alpha in alphas:
        ls = LabelSpreading(
            kernel=kernel, n_neighbors=n, gamma=g, alpha=alpha, max_iter=1000, tol=0.001
        )
        ls.fit(X_train, y_train)
        roc_scores.append(roc_auc_score(y_test, ls.predict_proba(X_test)[:, 1]))
    plt.figure(figsize=(16, 8))
    plt.plot(alphas, roc_scores)
    plt.title("Label Spreading ROC AUC with " + kernel + " kernel")
    plt.show()
    print("Best metrics value is at {}".format(alphas[np.argmax(roc_scores)]))

In [ ]:
alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
labels_spread_test("rbf", 1e-5, alphas, X_1_2, X_3, y_1_2, y_3)

In [ ]:
alphas = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09]
labels_spread_test("knn", 53, alphas, X_1_2, X_3, y_1_2, y_3)

In [ ]:
ls_rbf = LabelSpreading(kernel="rbf", gamma=9e-6, alpha=0.6, max_iter=1000, tol=0.001)
ls_rbf.fit(X_1_2, y_1_2)
results = results.append(
    pd.Series(
        ["Label Spreading RBF", roc_auc_score(y_3, ls_rbf.predict_proba(X_3)[:, 1])],
        index=index,
    ),
    ignore_index=True,
)
ls_knn = LabelSpreading(
    kernel="knn", n_neighbors=53, alpha=0.08, max_iter=1000, tol=0.001
)
ls_knn.fit(X_1_2, y_1_2)
results = results.append(
    pd.Series(
        ["Label Spreading KNN", roc_auc_score(y_3, ls_knn.predict_proba(X_3)[:, 1])],
        index=index,
    ),
    ignore_index=True,
)

In [ ]:
results


والتوقيتات. هم نفس، كما LP.


In [ ]:
%%timeit
knn_ls_test = LabelSpreading(kernel="rbf")
knn_ls_test.fit(df, y)
knn_ls_test.predict_proba(df);

In [ ]:
%%timeit
knn_ls_test = LabelSpreading(kernel="knn")
knn_ls_test.fit(df, y)
knn_ls_test.predict_proba(df);


## ساذج بايز



الآن دعنا ننتقل إلى `pomegranate` lib. وهو يدعم SSL لبعض النماذج: Naive Bayes، Bayes Classificator، Hidden Markov Models.
سوف نستخدم Naive Bayes (NB) فقط، الأمر سهل وبسيط.
>مصنفات Naive Bayes هي عائلة من "المصنفات الاحتمالية" البسيطة التي تعتمد على تطبيق نظرية بايز مع افتراضات استقلالية قوية (ساذجة) بين الميزات [[المصدر]](https://en.wikipedia.org/wiki/Naive_Bayes_classifier)
<center><p><img src="@@KEEP_00051@@ title="NB explanation"/></p></center>
سوف نستخدم [نظرية بايز](https://en.wikipedia.org/wiki/Bayes%27_theorem) لتوقعاتنا. لذلك، نحن بحاجة إلى العثور على $p(c|x)$ (احتمال الفئة c بالنظر إلى العينة x). للقيام بذلك، نحتاج إلى حساب $p(c)$ - مجرد احتمال فئة بشكل عام، $p(x)$ - *الدليل* الخاص بنا و$p(x|c)$. من الصعب جدًا حساب الأخير لأننا نحتاج بشكل عام إلى مراعاة التبعيات الشرطية في بياناتنا.
ولكن ماذا لو افترضنا أن جميع ميزاتنا مستقلة بشكل مشروط بالنظر إلى الفئة ج (افتراض قوي جدًا وخاطئ في الغالب)؟ حسنًا، هذا من شأنه أن يجعل حياتنا **كثيرًا** أسهل.
يمكننا الآن حساب $p(c|x)$ كمنتج لـ $p(x_i|c)$.


<center><p><img src="@@KEEP_00053@@ title="NB explanation 2"/></p></center>
Img 3 - شرح ساذج لبايز [[المصدر]](https://www.saedsayad.com/images/Bayes_rule.png)



لكل عينة سنختار الفئة الأكثر احتمالا (وهذا ما يعرف بالحد الأقصى لقاعدة القرار البعدي أو MAP). ثم يمكننا كتابة مهمة التحسين لدينا



<center><p><img src="@@KEEP_00055@@ title="NB optimization"/></p></center>
Img 4 - مُصنف Naive Bayes [[المصدر]](https://towardsdatascience.com/introduction-to-naive-bayes-classification-4cffabb1ae54)



الافتراض الآخر الذي يستفيد منه Naive Bayes هو المساواة بين جميع ميزاتنا. وهذا خطأ أيضًا، ولكن معه لا نحتاج إلى تحديد أوزان لمميزاتنا. على الرغم من أن بعض libs تمنحك هذا الاحتمال.
هناك الكثير من المقالات حول NB، أحب هذه المقالات [واحدة](https://jakevdp.github.io/PythonDataScienceHandbook/05.05-naive-bayes.html) و[واحدة من الويكي](https://en.wikipedia.org/wiki/Naive_Bayes_classifier).
`pomegranate` مستندات مستخدم مصنف bayes موجودة [هنا](https://pomegranate.readthedocs.io/en/latest/NaiveBayes.html)



الآن سنبدأ نموذج NB الخاص بنا مباشرةً من بياناتنا. `from_samples` هي الطريقة التي تسمح لنا بالقيام بذلك وتعيين التوزيع الذي سيشكل بياناتنا.
هناك طريقة أخرى تتمثل في التهيئة المسبقة للتوزيعات ومعلماتها مباشرةً ثم التنبؤ بفئات العينات.
لقد اخترت هنا `ExponentialDistribution` ولكن يمكنك اختيار ما تريد. قائمة التوزيعات المدعومة موجودة [هنا](https://pomegranate.readthedocs.io/en/latest/Distributions.html). العناصر التي تريدها هنا هي: `ExponentialDistribution`، `NormalDistribution`، `PoissonDistribution`. يمكنك التحقق من الآخرين ولكنه سيعطيك خطأ لأن الخوارزمية لن تتقارب.


In [ ]:
nb = pg.NaiveBayes.from_samples(pg.ExponentialDistribution, X_1_2, y_1_2, verbose=True)
roc_auc_score(y_3, nb.predict_proba(X_3)[:, 1])


قريبة من العشوائية. يشعر بالسوء. دعونا نجرب بعض الأشياء المجنونة. سنقوم ببناء توزيع المكونات المستقل الخاص بنا بأبعاد n (في هذه الحالة 7). سنقوم بالغش قليلاً، لأننا نعرف بالفعل كيفية توزيع بياناتنا (راجع `sns.pairplot` الخطوة). وجهة نظري في هذا أدناه


In [ ]:
d = [
    pg.ExponentialDistribution,
    pg.PoissonDistribution,
    pg.NormalDistribution,
    pg.ExponentialDistribution,
    pg.ExponentialDistribution,
    pg.PoissonDistribution,
    pg.NormalDistribution,
]
nb = pg.NaiveBayes.from_samples(d, X_1_2, y_1_2, verbose=True)
results = results.append(
    pd.Series(
        ["Naive Bayes ICD Prior", roc_auc_score(y_3, nb.predict_proba(X_3)[:, 1])],
        index=index,
    ),
    ignore_index=True,
)

In [ ]:
results

لا يزال الأمر سيئًا نظرًا لأن بياناتنا مترابطة وغير متساوية على الإطلاق. على الأقل `logreg` لا يعتقد ذلك.
ولكن لا يزال من الممكن أن يكون NB مفيدًا في بعض الحالات نظرًا لأنه بسيط وسريع جدًا وقابل للتفسير. يمكنك استخدامه كخط أساس أو كخوارزمية "استكشافية".


In [ ]:
plt.hist(logreg.coef_.reshape(7, 1));


أخيرًا وليس آخرًا - التوقيت (كما قلت - سريع جدًا، حتى في حالة البيانات المتفرقة)


In [ ]:
%%timeit
nb_test = pg.NaiveBayes.from_samples(pg.ExponentialDistribution, df, y, verbose=False)
nb_test.predict_proba(df);


##مكافأة



والأخيرة وليس الأخيرة. وضع العلامات الزائفة. لن أقوم بنسخه ولصقه، يمكنك القيام بذلك بنفسك. هذا هو النهج البسيط جدا. سنقوم فقط بفصل بياناتنا المُصنفة وغير المُسماة، وتدريب النموذج على المُسمى. بعد ذلك، سنقوم بأخذ عينات من البيانات غير المسماة ونتوقع هذه العينات ونضيفها إلى البيانات المصنفة كحقيقة أساسية جديدة. هذا كل شيء.
يمكنك حرفيًا استخدام كل شيء معها: نماذج الانحدار أو نماذج التصنيف. في الواقع، تم تصميمه للشبكات العصبية، لكنه متعدد الاستخدامات للغاية.



<center><p><img src="@@KEEP_00061@@ title="Pseudo-Labeling"/></p></center>
Img 5 - شرح العلامات الزائفة [[المصدر]](https://datawhatnow.com/pseudo-labeling-semi-supervised-learning/)



تم وصف العلامات الزائفة [هنا](https://datawhatnow.com/pseudo-labeling-semi-supervised-learning/) مع التعليمات البرمجية والرسوم البيانية، [نسخ ولصق على مدونة أخرى](https://www.analyticsvidhya.com/blog/2017/09/pseudo-labelling-semi-supervised-learning-technique/).
المقال الأصلي موجود [هنا](http://deeplearning.net/wp-content/uploads/2013/03/pseudo_label_final.pdf)
يمكنك نسخه أدناه واللعب به!